In [2]:
import os
import pandas as pd
import numpy as np

In [3]:
%pwd

'd:\\ML_AI\\PROJECTS\\TMDB_API_PROJECT\\notebooks'

In [4]:
os.chdir("../")
%pwd

'd:\\ML_AI\\PROJECTS\\TMDB_API_PROJECT'

In [5]:
data = pd.read_csv("data/raw/movies.csv")
data.head()

,id,title,overview,genres,vote_avg,release_date,popularity,budget,revenue
0,1290821,Shelter,A man living in self-imposed exile on a remote...,"[28, 80, 53]",6.760,2026-01-28,372.8761,50000000,42079609
1,10156,History of the World: Part I,An uproarious version of history that proves n...,[35],6.757,1981-06-12,212.8477,10000000,31672907
2,680493,Return to Silent Hill,When James receives a mysterious letter from h...,"[9648, 18, 27]",5.000,2026-01-21,164.5405,23000000,43023897
3,1159559,Scream 7,When a new Ghostface killer emerges in the qui...,"[27, 9648, 80]",6.000,2026-02-25,163.9871,45000000,112649685
4,1236153,Mercy,"In the near future, a detective stands on tria...","[878, 28, 53]",7.076,2026-01-20,132.3363,60000000,54310041


In [ ]:
data.isna().sum()

id              0
title           0
overview        0
genres          0
vote_avg        0
release_date    0
popularity      0
budget          0
revenue         0
dtype: int64

In [ ]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 734 entries, 0 to 733
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            734 non-null    int64  
 1   title         734 non-null    str    
 2   overview      734 non-null    str    
 3   genres        734 non-null    str    
 4   vote_avg      734 non-null    float64
 5   release_date  734 non-null    str    
 6   popularity    734 non-null    float64
 7   budget        734 non-null    int64  
 8   revenue       734 non-null    int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 51.7 KB


No Null or Na values

In [ ]:
data.shape

(734, 9)

In [ ]:
#  Creating the lavel or target col
def create_label(row):
    hit = row['revenue'] > 2.5 * row['budget']
    hit = hit and (row['vote_avg'] >= 7.5)
    return int(hit)

data['label'] = data.apply(create_label, axis=1)
data['label'].value_counts(normalize=True) * 100

label
1    58.174387
0    41.825613
Name: proportion, dtype: float64

If we just take this parameter it's very imbalanced

In [ ]:
data.columns

Index(['id', 'title', 'overview', 'genres', 'vote_avg', 'release_date',
       'popularity', 'budget', 'revenue', 'label'],
      dtype='str')

In [ ]:
data.sample(5)

,id,title,overview,genres,vote_avg,release_date,popularity,budget,revenue,label
97,1110034,Kraken,Marine biologist Johanne is doing research on ...,"[27, 28, 53]",6.714,2026-02-06,34.7753,1000000,14329678,1
628,767,Harry Potter and the Half-Blood Prince,As Lord Voldemort tightens his grip on both th...,"[12, 14]",7.681,2009-07-15,15.8936,250000000,933959197,1
443,805,Rosemary's Baby,"A young couple, Rosemary and Guy, moves into a...","[18, 27, 53]",7.819,1968-06-12,8.0433,3200000,33395426,1
641,1908,Inherit the Wind,Schoolteacher Bertram Cates is arrested for te...,[18],7.670,1960-07-07,0.8835,2000000,2000000,0
570,1291608,Dhurandhar,A mysterious traveler slips into the heart of ...,"[28, 80, 53]",6.900,2025-12-05,19.2077,22500000,160000000,1


In [ ]:
data.shape

(734, 10)

In [ ]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[^a-z0-9\s]','', text.lower().strip())
    return text

data["clean_overview"] = data["overview"].apply(clean_text)
data["clean_title"] = data["title"].apply(clean_text)

data["text"] = data["clean_title"] + " " + data["clean_overview"]

In [ ]:
new_data = data.copy()
new_data.drop(columns=["id", "title", "overview", "clean_overview", "clean_title"], inplace=True)
new_data.head()

,genres,vote_avg,release_date,popularity,budget,revenue,label,text
0,"[28, 80, 53]",6.760,2026-01-28,372.8761,50000000,42079609,0,shelter a man living in selfimposed exile on a...
1,[35],6.757,1981-06-12,212.8477,10000000,31672907,1,history of the world part i an uproarious vers...
2,"[9648, 18, 27]",5.000,2026-01-21,164.5405,23000000,43023897,0,return to silent hill when james receives a my...
3,"[27, 9648, 80]",6.000,2026-02-25,163.9871,45000000,112649685,1,scream 7 when a new ghostface killer emerges i...
4,"[878, 28, 53]",7.076,2026-01-20,132.3363,60000000,54310041,0,mercy in the near future a detective stands on...


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=200, stop_words="english")
text_features = tfidf.fit_transform(new_data["text"])
text_features

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4504 stored elements and shape (734, 200)>

In [ ]:
num_cols = ['vote_avg', 'popularity', 'budget']

scaler = MinMaxScaler()
num_features = scaler.fit_transform(data[num_cols])
num_features

array([[5.24781341e-01, 1.00000000e+00, 1.07526720e-01],
       [5.24052478e-01, 5.70816058e-01, 2.15051996e-02],
       [9.71817298e-02, 4.41259839e-01, 4.94621939e-02],
       ...,
       [7.32021380e-01, 2.77758781e-02, 1.14255935e-03],
       [7.32021380e-01, 9.13489995e-03, 1.65589621e-02],
       [7.31778426e-01, 2.31777201e-02, 2.44980689e-04]], shape=(734, 3))

In [30]:
import scipy.sparse as sp

num_sparse = sp.csr_matrix(num_features)
X = sp.hstack([num_sparse, text_features])
y = data["label"].values

In [34]:
X.shape

(734, 203)

In [33]:
y.shape

(734,)